In [24]:
import pandas as pd    # to load dataset
import numpy as np     # for mathematic equation
from nltk.corpus import stopwords   # to get collection of stopwords
from sklearn.model_selection import train_test_split       # for splitting dataset
from tensorflow.keras.preprocessing.text import Tokenizer  # to encode text to int
from tensorflow.keras.preprocessing.sequence import pad_sequences   # to do padding or truncating
from tensorflow.keras.models import Sequential     # the model
from tensorflow.keras.layers import Embedding, LSTM, Dense # layers of the architecture
from tensorflow.keras.callbacks import ModelCheckpoint   # save model
from tensorflow.keras.models import load_model   # load saved model
import nltk
import re

In [7]:
data = pd.read_csv('/content/IMDB Dataset.csv')

print(data)


                                                  review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production. <br /><br />The...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically there's a family where a little boy ...  negative
4      Petter Mattei's "Love in the Time of Money" is...  positive
...                                                  ...       ...
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

[50000 rows x 2 columns]


In [8]:
nltk.download('stopwords')
english_stops = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [9]:
def load_dataset():
    df = pd.read_csv('IMDB Dataset.csv')
    x_data = df['review']       # Reviews/Input
    y_data = df['sentiment']    # Sentiment/Output

    # PRE-PROCESS REVIEW
    x_data = x_data.replace({'<.*?>': ''}, regex = True)          # remove html tag
    x_data = x_data.replace({'[^A-Za-z]': ' '}, regex = True)     # remove non alphabet
    x_data = x_data.apply(lambda review: [w for w in review.split() if w not in english_stops])  # remove stop words
    x_data = x_data.apply(lambda review: [w.lower() for w in review])   # lower case

    # ENCODE SENTIMENT -> 0 & 1
    y_data = y_data.replace('positive', 1)
    y_data = y_data.replace('negative', 0)

    return x_data, y_data

x_data, y_data = load_dataset()

print('Reviews')
print(x_data, '\n')
print('Sentiment')
print(y_data)

Reviews
0        [one, reviewers, mentioned, watching, oz, epis...
1        [a, wonderful, little, production, the, filmin...
2        [i, thought, wonderful, way, spend, time, hot,...
3        [basically, family, little, boy, jake, thinks,...
4        [petter, mattei, love, time, money, visually, ...
                               ...                        
49995    [i, thought, movie, right, good, job, it, crea...
49996    [bad, plot, bad, dialogue, bad, acting, idioti...
49997    [i, catholic, taught, parochial, elementary, s...
49998    [i, going, disagree, previous, comment, side, ...
49999    [no, one, expects, star, trek, movies, high, a...
Name: review, Length: 50000, dtype: object 

Sentiment
0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 50000, dtype: int64


/tmp/ipython-input-4067306172.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_data = y_data.replace('negative', 0)


In [10]:
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size = 0.2)

print('Train Set')
print(x_train, '\n')
print(x_test, '\n')
print('Test Set')
print(y_train, '\n')
print(y_test)

Train Set
47680    [was, meant, comedy, serious, drama, this, fil...
38701    [my, girls, love, show, we, stumbled, across, ...
39106    [roll, roll, it, big, gay, bruce, big, gay, de...
34491    [western, union, something, forgotten, classic...
45151    [by, time, movie, came, director, mark, lester...
                               ...                        
17344    [roman, polanski, masterfully, directs, sort, ...
20189    [when, john, carpenter, masterly, version, the...
19748    [this, probably, greatest, war, film, certainl...
39552    [it, always, certain, mixing, comedians, toget...
12973    [why, spend, moment, slogging, awkward, self, ...
Name: review, Length: 40000, dtype: object 

33403    [the, ruth, snyder, judd, gray, murder, inspir...
5924     [okay, make, mistake, pretty, awful, film, i, ...
24939    [the, film, much, potential, developed, mark, ...
31144    [the, truth, film, based, harold, robbins, nov...
11557    [it, sad, lucian, pintilie, stop, making, movi...
 

In [11]:
def get_max_length():
    review_length = []
    for review in x_train:
        review_length.append(len(review))

    return int(np.ceil(np.mean(review_length)))

In [12]:
# ENCODE REVIEW
token = Tokenizer(lower=False)    # no need lower, because already lowered the data in load_data()
token.fit_on_texts(x_train)
x_train = token.texts_to_sequences(x_train)
x_test = token.texts_to_sequences(x_test)

max_length = get_max_length()

x_train = pad_sequences(x_train, maxlen=max_length, padding='post', truncating='post')
x_test = pad_sequences(x_test, maxlen=max_length, padding='post', truncating='post')

total_words = len(token.word_index) + 1   # add 1 because of 0 padding

print('Encoded X Train\n', x_train, '\n')
print('Encoded X Test\n', x_test, '\n')
print('Maximum review length: ', max_length)

Encoded X Train
 [[ 1518   923   108 ...     0     0     0]
 [  214   415    40 ...     0     0     0]
 [ 1611  1611     7 ...    47    12  6867]
 ...
 [    8   143   689 ...     0     0     0]
 [    7   124   662 ... 18644   576 21580]
 [  359  1083   468 ...   996   495 13388]] 

Encoded X Test
 [[    2  4116 38031 ... 13140   945   225]
 [  727    25  1288 ...     0     0     0]
 [    2     4    17 ...     0     0     0]
 ...
 [    1  8652 24255 ...     0     0     0]
 [    1   241   670 ...     0     0     0]
 [    8     3    26 ...     0     0     0]] 

Maximum review length:  130


In [13]:
# ARCHITECTURE
EMBED_DIM = 32
LSTM_OUT = 64

model = Sequential()
model.add(Embedding(total_words, EMBED_DIM, input_length = max_length))
model.add(LSTM(LSTM_OUT))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

print(model.summary())

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [14]:
checkpoint = ModelCheckpoint(
    'models/LSTM.h5',
    monitor='accuracy',
    save_best_only=True,
    verbose=1
)

In [15]:
model.fit(x_train, y_train, batch_size = 128, epochs = 5, callbacks=[checkpoint])

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step - accuracy: 0.5310 - loss: 0.6819
Epoch 1: accuracy improved from -inf to 0.56450, saving model to models/LSTM.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 69s 211ms/step - accuracy: 0.5311 - loss: 0.6818
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step - accuracy: 0.5929 - loss: 0.6504
Epoch 2: accuracy improved from 0.56450 to 0.59555, saving model to models/LSTM.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 65s 207ms/step - accuracy: 0.5929 - loss: 0.6505
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step - accuracy: 0.6485 - loss: 0.6333
Epoch 3: accuracy improved from 0.59555 to 0.65820, saving model to models/LSTM.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 81s 203ms/step - accuracy: 0.6485 - loss: 0.6332
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step - accuracy: 0.6650 - loss: 0.6281
Epoch 4: accuracy improved from 0.65820 to 0.68257, saving model to models/LSTM.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 83s 206ms/step - accuracy: 0.6650 - loss: 0.6281
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step - accuracy: 0.7509 - loss: 0.5262
Epoch 5: accuracy improved from 0.68257 to 0.75738, saving model to models/LSTM.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 82s 207ms/step - accuracy: 0.7509 - loss: 0.5262


In [18]:
y_pred_prob = model.predict(x_test, batch_size = 128)
y_pred = (y_pred_prob > 0.5).astype(int) # Convert probabilities to binary predictions

true = 0
for i, y in enumerate(y_test):
    if y == y_pred[i]:
        true += 1

print('Correct Prediction: {}'.format(true))
print('Wrong Prediction: {}'.format(len(y_pred) - true))
print('Accuracy: {}'.format(true/len(y_pred)*100))

79/79 ━━━━━━━━━━━━━━━━━━━━ 6s 71ms/step
Correct Prediction: 7642
Wrong Prediction: 2358
Accuracy: 76.42


In [19]:
loaded_model = load_model('models/LSTM.h5')

In [36]:
review = str(input('Movie Review: '))

Movie Review: Everything was beautifully done in this movie, the story, the flow, the scenario, everything. I highly recommend it for mystery lovers, for anyone who wants to watch a good movie!


In [37]:
# Pre-process input
regex = re.compile(r'[^a-zA-Z\s]')
review = regex.sub('', review)
print('Cleaned: ', review)

words = review.split(' ')
filtered = [w for w in words if w not in english_stops]
filtered = ' '.join(filtered)
filtered = [filtered.lower()]

print('Filtered: ', filtered)

Cleaned:  Everything was beautifully done in this movie the story the flow the scenario everything I highly recommend it for mystery lovers for anyone who wants to watch a good movie
Filtered:  ['everything beautifully done movie story flow scenario everything i highly recommend mystery lovers anyone wants watch good movie']


In [38]:
tokenize_words = token.texts_to_sequences(filtered)
tokenize_words = pad_sequences(tokenize_words, maxlen=max_length, padding='post', truncating='post')
print(tokenize_words)

[[ 173 1181  127    3   15 2797 2619  173    1  454  285  693 1680  152
   398   33    9    3    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0]]


In [39]:
result = loaded_model.predict(tokenize_words)
print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
[[0.8552616]]


In [40]:
if result >= 0.7:
    print('positive')
else:
    print('negative')

positive
